<a href="https://colab.research.google.com/github/fishmoonbird/NN/blob/main/00_environment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("⚠️ 当前没有检测到 GPU，请检查 Colab Runtime 设置")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [6]:
!pip -q install datasets tokenizers transformers

In [7]:
import os
import random
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Imports successful.")

Imports successful.


In [8]:
SEED = 42

def set_seed(seed=42):
    """
    固定各种随机数生成器，
    尽可能让每次实验结果可复现。
    """

    # Python 自己的随机数
    random.seed(seed)

    # NumPy 随机数
    np.random.seed(seed)

    # PyTorch CPU 随机数
    torch.manual_seed(seed)

    # PyTorch GPU 随机数
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 尽量让 CUDA 的计算结果可复现
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

print("Random seed:", SEED)

Random seed: 42


In [9]:
SEED = 42

def set_seed(seed=42):
    """
    固定各种随机数生成器，
    尽可能让每次实验结果可复现。
    """

    # Python 自己的随机数
    random.seed(seed)

    # NumPy 随机数
    np.random.seed(seed)

    # PyTorch CPU 随机数
    torch.manual_seed(seed)

    # PyTorch GPU 随机数
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 尽量让 CUDA 的计算结果可复现
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

print("Random seed:", SEED)

Random seed: 42


In [10]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cuda


In [11]:
PROJECT_ROOT = Path("/content/transformer_lab")

DATA_DIR = PROJECT_ROOT / "data"
TOKENIZER_DIR = PROJECT_ROOT / "tokenizer"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# 创建文件夹
for directory in [
    DATA_DIR,
    TOKENIZER_DIR,
    CHECKPOINT_DIR,
    OUTPUT_DIR,
    CHECKPOINT_DIR / "transformer",
    CHECKPOINT_DIR / "bert",
    CHECKPOINT_DIR / "gpt",
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /content/transformer_lab


In [12]:
CONFIG = {
    # -------------------------
    # Tokenizer
    # -------------------------
    "vocab_size": 8000,

    # -------------------------
    # 输入序列
    # -------------------------
    "max_seq_len": 128,

    # -------------------------
    # Transformer 模型大小
    # -------------------------
    "d_model": 256,
    "num_heads": 4,
    "num_layers": 4,
    "d_ff": 1024,
    "dropout": 0.1,

    # -------------------------
    # Training
    # -------------------------
    "batch_size": 32,
    "learning_rate": 3e-4,
    "epochs": 3,

    # -------------------------
    # Reproducibility
    # -------------------------
    "seed": SEED,
}

CONFIG

{'vocab_size': 8000,
 'max_seq_len': 128,
 'd_model': 256,
 'num_heads': 4,
 'num_layers': 4,
 'd_ff': 1024,
 'dropout': 0.1,
 'batch_size': 32,
 'learning_rate': 0.0003,
 'epochs': 3,
 'seed': 42}

In [13]:
x = torch.randn(1000, 1000).to(device)
y = torch.randn(1000, 1000).to(device)

z = x @ y

print("x shape:", x.shape)
print("z shape:", z.shape)
print("z device:", z.device)

x shape: torch.Size([1000, 1000])
z shape: torch.Size([1000, 1000])
z device: cuda:0


In [14]:
if torch.cuda.is_available():

    props = torch.cuda.get_device_properties(0)

    total_memory = props.total_memory / 1024**3

    print(f"GPU: {props.name}")
    print(f"Total GPU Memory: {total_memory:.2f} GB")

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    print(f"Allocated: {allocated:.3f} GB")
    print(f"Reserved:  {reserved:.3f} GB")

GPU: Tesla T4
Total GPU Memory: 14.56 GB
Allocated: 0.020 GB
Reserved:  0.020 GB


In [15]:
# 一个非常简单的神经网络
model = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 2)
).to(device)

# 随机生成一个 batch
x = torch.randn(32, 10).to(device)

# 假设是二分类问题
labels = torch.randint(
    low=0,
    high=2,
    size=(32,)
).to(device)

# -------------------------
# Forward
# -------------------------

logits = model(x)

print("logits shape:", logits.shape)

# -------------------------
# Loss
# -------------------------

loss = F.cross_entropy(
    logits,
    labels
)

print("loss:", loss.item())

# -------------------------
# Backward
# -------------------------

loss.backward()

print("Backward successful!")

logits shape: torch.Size([32, 2])
loss: 0.7185564637184143
Backward successful!


In [16]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

for step in range(5):

    # 每一步先清空上一轮梯度
    optimizer.zero_grad()

    # Forward
    logits = model(x)

    # 计算 loss
    loss = F.cross_entropy(
        logits,
        labels
    )

    # Backward：计算梯度
    loss.backward()

    # 根据梯度更新模型参数
    optimizer.step()

    print(
        f"Step {step + 1} | "
        f"Loss: {loss.item():.4f}"
    )

Step 1 | Loss: 0.7186
Step 2 | Loss: 0.7170
Step 3 | Loss: 0.7155
Step 4 | Loss: 0.7139
Step 5 | Loss: 0.7124
